# 01 ? Conversion Funnel Validation
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Reproduce and statistically validate the conversion funnel metrics using Python and Pandas against DuckDB.

---
### Key Questions:
1. What are the exact session counts and drop-offs across the core e-commerce stages?
2. How does the funnel differ between Search-engaged sessions and Browse-only sessions?
3. Are the conversion rate differences statistically robust with 95% confidence intervals?


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from exploratory_analysis import get_db_connection, calc_proportion_ci, two_proportion_z_test

con = get_db_connection()
print("Connected to DuckDB successfully.")


## 1. End-to-End Session Funnel Calculation
Calculating distinct sessions reaching each major funnel milestone:
- **Step 1 (Total Sessions):** All recorded sessions
- **Step 2 (PDP View):** Sessions with at least 1 product view
- **Step 3 (Add to Cart):** Sessions with at least 1 cart item addition
- **Step 4 (Order Placed):** Sessions with a completed order


In [ ]:
funnel_query = '''
SELECT 
    COUNT(DISTINCT s.session_id) AS total_sessions,
    COUNT(DISTINCT pv.session_id) AS pdp_sessions,
    COUNT(DISTINCT ce.session_id) AS cart_sessions,
    COUNT(DISTINCT o.session_id) AS order_sessions
FROM sessions s
LEFT JOIN product_views pv ON s.session_id = pv.session_id
LEFT JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
'''
df_funnel = con.execute(funnel_query).df()

tot = df_funnel['total_sessions'][0]
pdp = df_funnel['pdp_sessions'][0]
crt = df_funnel['cart_sessions'][0]
ord_cnt = df_funnel['order_sessions'][0]

metrics = [
    {'Stage': '1. Total Sessions', 'Sessions (N)': tot, 'Step Conv (%)': 100.0, 'Drop-off (%)': 0.0, 'Overall Conv (%)': 100.0, '95% CI': 'N/A'},
    {'Stage': '2. PDP View', 'Sessions (N)': pdp, 'Step Conv (%)': round(pdp/tot*100, 2), 'Drop-off (%)': round((1 - pdp/tot)*100, 2), 'Overall Conv (%)': round(pdp/tot*100, 2),
     '95% CI': f"[{calc_proportion_ci(pdp, tot)[1]*100:.2f}%, {calc_proportion_ci(pdp, tot)[2]*100:.2f}%]"},
    {'Stage': '3. Add to Cart', 'Sessions (N)': crt, 'Step Conv (%)': round(crt/pdp*100, 2), 'Drop-off (%)': round((1 - crt/pdp)*100, 2), 'Overall Conv (%)': round(crt/tot*100, 2),
     '95% CI': f"[{calc_proportion_ci(crt, pdp)[1]*100:.2f}%, {calc_proportion_ci(crt, pdp)[2]*100:.2f}%]"},
    {'Stage': '4. Order Placed', 'Sessions (N)': ord_cnt, 'Step Conv (%)': round(ord_cnt/crt*100, 2), 'Drop-off (%)': round((1 - ord_cnt/crt)*100, 2), 'Overall Conv (%)': round(ord_cnt/tot*100, 2),
     '95% CI': f"[{calc_proportion_ci(ord_cnt, crt)[1]*100:.2f}%, {calc_proportion_ci(ord_cnt, crt)[2]*100:.2f}%]"}
]

summary_df = pd.DataFrame(metrics)
print("Validated Session Funnel:")
display(summary_df) if 'display' in dir() else print(summary_df.to_string())


## 2. Search vs Browse Funnel Segmentation
Testing the hypothesis: Do search-engaged sessions exhibit a higher downstream conversion propensity?


In [ ]:
search_funnel_q = '''
SELECT 
    CASE WHEN s.has_search THEN 'Search Sessions' ELSE 'Browse Sessions' END AS session_segment,
    COUNT(DISTINCT s.session_id) AS total_sessions,
    COUNT(DISTINCT pv.session_id) AS pdp_sessions,
    ROUND(100.0 * COUNT(DISTINCT pv.session_id) / COUNT(DISTINCT s.session_id), 2) AS s_to_pdp_pct,
    COUNT(DISTINCT ce.session_id) AS cart_sessions,
    ROUND(100.0 * COUNT(DISTINCT ce.session_id) / COUNT(DISTINCT pv.session_id), 2) AS pdp_to_cart_pct,
    COUNT(DISTINCT o.session_id) AS order_sessions,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT ce.session_id), 2) AS cart_to_order_pct,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT s.session_id), 2) AS session_to_order_pct
FROM sessions s
LEFT JOIN product_views pv ON s.session_id = pv.session_id
LEFT JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
GROUP BY 1
'''
df_sb = con.execute(search_funnel_q).df()
print("Search vs Browse Funnel Performance:")
print(df_sb.to_string())

# Hypothesis test on overall conversion
s_ord, s_tot = df_sb.loc[df_sb['session_segment']=='Search Sessions', ['order_sessions', 'total_sessions']].values[0]
b_ord, b_tot = df_sb.loc[df_sb['session_segment']=='Browse Sessions', ['order_sessions', 'total_sessions']].values[0]
diff, z, p, ci = two_proportion_z_test(s_ord, s_tot, b_ord, b_tot)
print(f"\nSearch Lift Z-Test: Diff = +{diff*100:.2f} pp, Z = {z:.2f}, p-value = {p:.2e}")
print(f"Search sessions convert at {s_ord/s_tot / (b_ord/b_tot):.2f}x the rate of browse sessions.")


## 3. Visual Funnel Verification
Loading and displaying the validated presentation funnel chart.


In [ ]:
from IPython.display import Image
Image(filename='../reports/figures/01_funnel_chart.png')
